In [3]:
import json
import pandas as pd
import serpapi

with open("../airports.json") as f:
    raw = json.load(f)

# raw is: {"00AK": {...}, "00AL": {...}, ...}
# pd.DataFrame.from_dict with orient="index" makes each key a row
df = pd.DataFrame.from_dict(raw, orient="index")
# Step 1: keep only Italy and Germany
df = df[df["country"].isin(["IT", "DE"])]

# Step 2: keep only airports that have an IATA code (commercial airports)
df = df[df["iata"] != ""]

# Step 3: reset index so row numbers are clean
df = df.reset_index(drop=True)

print(df)

     icao iata                                               name  \
0    EDAC  AOC                           Altenburg-Nobitz Airport   
1    EDAH  HDF                                Heringsdorf Airport   
2    EDAQ  ZHZ                                Halle-Oppin Airport   
3    EDAU  IES                               Riesa-Gohlis Airport   
4    EDAX  REB                               Rechlin-Larz Airport   
..    ...  ...                                                ...   
173  LIRN  NAP         Napoli / Capodichino International Airport   
174  LIRP  PSA  Pisa / San Giusto - Galileo Galilei Internatio...   
175  LIRQ  FLR                         Firenze / Peretola Airport   
176  LIRS  GRS                                   Grosseto Airport   
177  LIRZ  PEG                       Perugia / San Egidio Airport   

            city                   state country  elevation        lat  \
0      Altenburg               Thuringia      DE        640  50.981945   
1    Heringsdorf  Meckl

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   icao       178 non-null    str    
 1   iata       178 non-null    str    
 2   name       178 non-null    str    
 3   city       178 non-null    str    
 4   state      178 non-null    str    
 5   country    178 non-null    str    
 6   elevation  178 non-null    int64  
 7   lat        178 non-null    float64
 8   lon        178 non-null    float64
 9   tz         178 non-null    str    
dtypes: float64(2), int64(1), str(7)
memory usage: 14.0 KB


In [5]:
df.head()

,icao,iata,name,city,state,country,elevation,lat,lon,tz
0,EDAC,AOC,Altenburg-Nobitz Airport,Altenburg,Thuringia,DE,640,50.981945,12.506389,Europe/Berlin
1,EDAH,HDF,Heringsdorf Airport,Heringsdorf,Mecklenburg-Vorpommern,DE,93,53.878700,14.152300,Europe/Berlin
2,EDAQ,ZHZ,Halle-Oppin Airport,Oppin,Saxony-Anhalt,DE,348,51.552223,12.053889,Europe/Berlin
3,EDAU,IES,Riesa-Gohlis Airport,Riesa,Saxony,DE,322,51.293610,13.356111,Europe/Berlin
4,EDAX,REB,Rechlin-Larz Airport,Larz,Mecklenburg-Vorpommern,DE,220,53.306389,12.752222,Europe/Berlin


In [9]:
import difflib
name = input("Input the name of the airport: ")
matches = difflib.get_close_matches(name, df["name"], n=3)

In [10]:
matches

[]

In [11]:
from thefuzz import process
best_match = process.extract(name, df["name"], limit=5)

In [12]:
best_match


[('Frankfurt am Main International Airport', 90, 15),
 ('Frankfurt-Egelsbach Airport', 90, 29),
 ('Frankfurt-Hahn Airport', 90, 30),
 ('Verona / Villafranca Airport', 72, 160),
 ('Berlin Brandenburg Airport', 54, 12)]

In [13]:
df

,icao,iata,name,city,state,country,elevation,lat,lon,tz
0,EDAC,AOC,Altenburg-Nobitz Airport,Altenburg,Thuringia,DE,640,50.981945,12.506389,Europe/Berlin
1,EDAH,HDF,Heringsdorf Airport,Heringsdorf,Mecklenburg-Vorpommern,DE,93,53.878700,14.152300,Europe/Berlin
2,EDAQ,ZHZ,Halle-Oppin Airport,Oppin,Saxony-Anhalt,DE,348,51.552223,12.053889,Europe/Berlin
3,EDAU,IES,Riesa-Gohlis Airport,Riesa,Saxony,DE,322,51.293610,13.356111,Europe/Berlin
4,EDAX,REB,Rechlin-Larz Airport,Larz,Mecklenburg-Vorpommern,DE,220,53.306389,12.752222,Europe/Berlin
...,...,...,...,...,...,...,...,...,...,...
173,LIRN,NAP,Napoli / Capodichino International Airport,Napoli,Campania,IT,294,40.886002,14.290800,Europe/Rome
174,LIRP,PSA,Pisa / San Giusto - Galileo Galilei Internatio...,Pisa,Tuscany,IT,6,43.683899,10.392700,Europe/Rome
175,LIRQ,FLR,Firenze / Peretola Airport,Firenze,Tuscany,IT,142,43.810001,11.205100,Europe/Rome
176,LIRS,GRS,Grosseto Airport,Grosetto,Tuscany,IT,17,42.759701,11.071900,Europe/Rome


In [42]:
# travel_planner.py
import math
import json
from functools import lru_cache
from dataclasses import dataclass
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import os
import serpapi
from dotenv import load_dotenv
# ========================= CONFIG =========================
load_dotenv()
API_KEY = os.getenv("SERPAPI_KEY")  # ← put in .env, never hardcode
GEOLOCATOR = Nominatim(user_agent="travel_planner_kerem")  # ONE instance

COUNTRY_EN = {"DE": "Germany", "IT": "Italy"}  # extend as needed

# ========================= DATA =========================
with open("../airports.json") as f:  # put json next to script or use absolute path
    airports_df = (
        pd.DataFrame.from_dict(json.load(f), orient="index")
        .query("country in ['DE', 'IT'] and iata != ''")
        [["iata", "name", "city", "country", "lat", "lon"]]
        .reset_index(drop=True)
    )

# ========================= HELPERS =========================
@dataclass
class Location:
    city: str
    country: str
    lat: float
    lon: float
    iata: str | None = None
    bus_name: str | None = None  # "City, Country" for CheckMyBus

@lru_cache(maxsize=100)  # ← caches repeated geocoding calls!
def geocode_city(city: str) -> tuple[float, float]:
    try:
        loc = GEOLOCATOR.geocode(city, exactly_one=True, timeout=10)
        if loc is None:
            raise ValueError(f"Could not geocode '{city}'")
        return loc.latitude, loc.longitude
    except (GeocoderTimedOut, GeocoderServiceError) as e:
        raise ValueError(f"Geocoding error for '{city}': {e}") from e

@lru_cache(maxsize=100)
def city_en_from_coords(lat: float, lon: float, country_code: str, iata: str = "") -> str:
    """Reverse geocode → perfect CheckMyBus string (English)."""
    country = COUNTRY_EN.get(country_code, country_code)
    loc = GEOLOCATOR.reverse((lat, lon), language="en", addressdetails=True, exactly_one=True, timeout=10)
    if not loc:
        raise ValueError(f"Reverse geocode failed for ({lat}, {lon})")
    addr = loc.raw.get("address", {})
    city = addr.get("city") or addr.get("town") or addr.get("village") or addr.get("county")
    if not city:
        raise ValueError(f"No city in address for ({lat}, {lon})")
    return f"{city}, {country}"

def iata_to_location(iata: str) -> Location:
    row = airports_df[airports_df["iata"] == iata.upper()]
    if row.empty:
        raise ValueError(f"IATA {iata} not in airports.json")
    r = row.iloc[0]
    country = COUNTRY_EN.get(r["country"], r["country"])
    bus_name = f'{r["city"]}, {country}'
    return Location(city=r["city"], country=country, lat=r["lat"], lon=r["lon"], iata=iata, bus_name=bus_name)

# ========================= FLIGHTS =========================
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    d_phi = math.radians(lat2 - lat1)
    d_lam = math.radians(lon2 - lon1)
    a = math.sin(d_phi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(d_lam / 2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def explore_search(departure_id: str, outbound_date: str, target_city: str, max_distance_km: int = 300):
    """Cheap flights to cities near your target + IATA + bus-ready name."""
    lat, lon = geocode_city(target_city)
    target = {"lat": lat, "lon": lon}

    client = serpapi.Client(api_key=API_KEY)
    results = client.search({
        "engine": "google_travel_explore",
        "departure_id": departure_id,
        "currency": "EUR",
        "type": "2",
        "outbound_date": outbound_date,
        "no_cache": False,
    })

    rows = []
    for dest in results.get("destinations", []):
        if "flight_price" not in dest:
            continue
        iata = dest.get("destination_airport", {}).get("code")  # ← NEW!
        lat_d, lon_d = dest["gps_coordinates"]["latitude"], dest["gps_coordinates"]["longitude"]
        distance = haversine_km(lat_d, lon_d, target["lat"], target["lon"])

        if distance > max_distance_km:
            continue

        # Build clean bus name (prefer reverse geocode on GPS)
        try:
            bus_name = city_en_from_coords(lat_d, lon_d, country_code=dest.get("country", "")[:2] or "DE", iata=iata or "")
        except Exception:
            # fallback
            bus_name = f"{dest['name']}, {dest.get('country', 'Unknown')}"

        rows.append({
            "city": dest["name"],
            "country": dest.get("country"),
            "iata": iata,
            "price": dest["flight_price"],
            "duration": dest["flight_duration"],
            "airline": dest["airline"],
            "link": dest["link"],
            "lat": lat_d,
            "lon": lon_d,
            "distance_km": round(distance, 1),
            "bus_departure_name": bus_name,   # ← ready for CheckMyBus
        })

    df = pd.DataFrame(rows).sort_values("price").reset_index(drop=True)
    return df

# ========================= BUS / TRAIN =========================
from checkmybus import CheckMyBusClient, CheckMyBusSearchParams

def bus_train_transfer(departure_location: str, arrival_location: str, departure_date: str):
    """Return DataFrame of buses/trains. departure_location must be clean 'City, Country'."""
    client = CheckMyBusClient()

    print(f"🔎 Bus/Train search STARTING: {departure_location} → {arrival_location} on {departure_date}")

    try:
        search_result = client.search(CheckMyBusSearchParams(
            departure_location=departure_location,
            arrival_location=arrival_location,
            departure_date=departure_date,
        ))

        df = search_result.to_dataframe()          # renamed for clarity

        # Count buses and trains safely (in case column is missing)
        if "transport_mode" in df.columns:
            count_bus = len(df[df["transport_mode"] == "bus"])
            count_train = len(df[df["transport_mode"] == "train"])
        else:
            count_bus = count_train = 0
            print("⚠️  'transport_mode' column not found in result!")

        print(f"✅ Search DONE for {departure_location}. "
              f"Total: {len(df)} results → {count_bus} bus(es), {count_train} train(s)")

        if df.empty:
            print(f"⚠️  No buses or trains found from {departure_location} to {arrival_location}")

        return df

    except Exception as e:   # catches API errors, bad parameters, network issues, etc.
        print(f"❌ ERROR searching {departure_location} → {arrival_location}: {e}")
        # You can raise or return empty DataFrame depending on how strict you want it
        return pd.DataFrame()   # empty DF so the main pipeline doesn't crash

# ========================= MAIN PIPELINE (the simple way) =========================
def find_cheap_flight_plus_ground(departure_id: str,
                                  target_city: str,          # e.g. "Munich, Germany" or just "Munich"
                                  outbound_date: str,        # "2026-05-16"
                                  ground_date: str = None,   # defaults to same day
                                  max_distance_km: int = 300):
    if ground_date is None:
        ground_date = outbound_date

    # 1. Get cheap flights near target
    nearby = explore_search(departure_id, outbound_date, target_city, max_distance_km)
    if nearby is not None:
        print(f"Hey good news! Found {len(nearby)} nearby flights.")
    else:
        print("Oh sorry! There is no nearby flights.")
    for _, row in nearby.iterrows():
        print(f"From {row["city"]} and price is {row["price"]}")

    # 2. For each, search ground transport
    combined = []
    for _, row in nearby.iterrows():
        try:
            ground_df = bus_train_transfer(                  # I renamed the variable to "ground_df" for clarity
                departure_location=row["bus_departure_name"],
                arrival_location=target_city,
                departure_date=ground_date
            )
            if ground_df.empty or "price" not in ground_df.columns:
                continue

            # ── NEW: handle BOTH bus and train in one clean loop ──
            for mode in ["bus", "train"]:
                # Filter the right transport mode (safe if column doesn't exist yet)
                mode_df = ground_df[ground_df.get("transport_mode", pd.Series(["bus"] * len(ground_df))) == mode]

                if not mode_df.empty:
                    best = mode_df.loc[mode_df["price"].idxmin()]   # cheapest in this mode

                    combined.append({
                        **row.to_dict(),                     # all flight info
                        "ground_type": mode.capitalize(),    # "Bus" or "Train"
                        "ground_price": best["price"],
                        "total_price": row["price"] + best["price"],
                        "ground_duration": best.get("duration_min", best.get("duration", None)),
                        "ground_link": best.get("checkmybus_url", best.get("link", None)),
                        "ground_departure": best.get("departure_dt", best.get("departure_time", None)),
                        "ground_arrival": best.get("arrival_dt", best.get("arrival_time", None)),
                    })
        except Exception as e:
            print(f"Bus search failed for {row['city']}: {e}")  # or log

    if combined:
        result_df = pd.DataFrame(combined).sort_values("total_price").reset_index(drop=True)
        return result_df[["city", "iata", "price", "ground_type", "ground_price", "total_price",
                          "distance_km", "bus_departure_name", "link",
                          "ground_duration", "ground_departure", "ground_arrival"]]
    else:
        return pd.DataFrame()  # empty



In [43]:
# ========================= USAGE EXAMPLE =========================
if __name__ == "__main__":
    df = find_cheap_flight_plus_ground(
        departure_id="VCE",
        target_city="Nürnberg",
        outbound_date="2026-05-10",
        ground_date="2026-05-10"
    )


Hey good news! Found 4 nearby flights.
From Frankfurt am Main and price is 81
From Prague and price is 102
From Zürich and price is 122
From Munich and price is 129
🔎 Bus/Train search STARTING: Frankfurt, Ge → Nürnberg on 2026-05-10
✅ Search DONE for Frankfurt, Ge. Total: 109 results → 15 bus(es), 94 train(s)
🔎 Bus/Train search STARTING: Prague, Cz → Nürnberg on 2026-05-10
✅ Search DONE for Prague, Cz. Total: 39 results → 32 bus(es), 7 train(s)
🔎 Bus/Train search STARTING: Zurich, Sw → Nürnberg on 2026-05-10
⚠️  'transport_mode' column not found in result!
✅ Search DONE for Zurich, Sw. Total: 0 results → 0 bus(es), 0 train(s)
⚠️  No buses or trains found from Zurich, Sw to Nürnberg
🔎 Bus/Train search STARTING: Munich, Ge → Nürnberg on 2026-05-10
✅ Search DONE for Munich, Ge. Total: 127 results → 19 bus(es), 107 train(s)


In [44]:
df

,city,iata,price,ground_type,ground_price,total_price,distance_km,bus_departure_name,link,ground_duration,ground_departure,ground_arrival
0,Frankfurt am Main,FRA,81,Bus,11.900000,92.900000,186.8,"Frankfurt, Ge",https://www.google.com/travel/explore?hl=en&gl...,190,2026-05-10 22:00:00,2026-05-11 01:10:00
1,Frankfurt am Main,FRA,81,Train,14.990000,95.990000,186.8,"Frankfurt, Ge",https://www.google.com/travel/explore?hl=en&gl...,124,2026-05-10 04:54:00,2026-05-10 06:58:00
2,Prague,PRG,102,Bus,17.999765,119.999765,251.0,"Prague, Cz",https://www.google.com/travel/explore?hl=en&gl...,215,2026-05-10 22:45:00,2026-05-11 02:20:00
3,Prague,PRG,102,Train,18.810000,120.810000,251.0,"Prague, Cz",https://www.google.com/travel/explore?hl=en&gl...,442,2026-05-10 17:35:00,2026-05-11 00:57:00
4,Munich,MUC,129,Bus,10.480000,139.480000,151.2,"Munich, Ge",https://www.google.com/travel/explore?hl=en&gl...,125,2026-05-10 06:05:00,2026-05-10 08:10:00
5,Munich,MUC,129,Train,14.990000,143.990000,151.2,"Munich, Ge",https://www.google.com/travel/explore?hl=en&gl...,76,2026-05-10 05:11:00,2026-05-10 06:27:00


In [35]:
df["link"]

0    https://www.google.com/travel/explore?hl=en&gl...
1    https://www.google.com/travel/explore?hl=en&gl...
2    https://www.google.com/travel/explore?hl=en&gl...
Name: link, dtype: str